# Module 37: Graded Mini Project - Conditional GAN

            **Student:** Manoj Bhardwaj

            **Goal:** Build a Conditional Generative Adversarial Network (cGAN) to generate 64x64 RGB images of cats or dogs conditioned on class labels.

            **Required label convention:**

            | Label | Class |
            |---:|---|
            | 0 | Cat |
            | 1 | Dog |


## Phase 1 - Data Preparation

            The dataset is loaded from TensorFlow Datasets using `cats_vs_dogs`. Images are resized to 64x64, normalized to `[-1, 1]`, labels are cast to integer tensors, and the training pipeline uses shuffle, batch, and prefetch.


In [ ]:
import sys
            from pathlib import Path

            PROJECT_ROOT = Path.cwd()
            if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
                PROJECT_ROOT = PROJECT_ROOT.parent
            if str(PROJECT_ROOT) not in sys.path:
                sys.path.insert(0, str(PROJECT_ROOT))

            from src.cgan_cats_vs_dogs import (
                CGANConfig,
                build_discriminator,
                build_generator,
                label_names_from_info,
                load_raw_datasets,
                prepare_dataset,
                save_real_sample_grid,
                train_cgan,
            )

            config = CGANConfig(epochs=10, batch_size=128, output_dir=Path("outputs"))
            DATA_DIR = ".tfds"
            VERIFY_SSL = False  # Set True if your local certificates allow TFDS downloads.

            ds_train_raw, ds_test_raw, ds_info = load_raw_datasets(
                data_dir=DATA_DIR,
                verify_ssl=VERIFY_SSL,
            )
            class_names = label_names_from_info(ds_info)
            print(f"Verified label mapping: 0 = {class_names[0]}, 1 = {class_names[1]}")

            train_dataset = prepare_dataset(ds_train_raw, config, training=True)
            test_dataset = prepare_dataset(ds_test_raw, config, training=False)
            save_real_sample_grid(
                test_dataset,
                class_names,
                Path("outputs/samples/real_preprocessed_samples.png"),
                config,
            )


## Phase 2 - Conditional GAN Architecture

            The generator accepts random noise and a class label. The label is embedded and concatenated with the noise vector before transposed convolution layers generate a `64x64x3` RGB image.

            The discriminator accepts an image and a class label. The label is embedded into a spatial `64x64x1` map and concatenated with the image before Conv2D layers classify the input as real or fake.


In [ ]:
generator = build_generator(config)
            discriminator = build_discriminator(config)

            generator.summary()
            discriminator.summary()


## Phase 3 and 4 - Losses, Optimizers, and Training

            The implementation uses `BinaryCrossentropy(from_logits=True)`. The generator is trained to make generated images look real to the discriminator. The discriminator is trained to classify real images as real and generated images as fake. Separate Adam optimizers are used for the two networks.


In [ ]:
history, generator, discriminator = train_cgan(
                config,
                data_dir=DATA_DIR,
                verify_ssl=VERIFY_SSL,
                max_train_batches=None,
                sample_every=1,
            )

            history


## Phase 5 - Evaluation and Visualization

            The training script saves a fixed class-conditioned image grid after each epoch. The first half of the grid is conditioned on label `0` (Cat), and the second half is conditioned on label `1` (Dog). Generated tensors are converted from `[-1, 1]` back to `[0, 1]` before display.

            Review these files after the run:

            - `outputs/samples/epoch_000_untrained.png`
            - `outputs/samples/epoch_001.png` through `outputs/samples/epoch_010.png`
            - `outputs/samples/real_preprocessed_samples.png`


## Results Interpretation

            After a 10-epoch training run, evaluate:

            - **Image quality:** whether generated samples show recognizable cat/dog-like shapes, colors, and texture.
            - **Diversity:** whether samples differ from each other instead of collapsing to a single repeated pattern.
            - **Label conditioning:** whether outputs conditioned on `Cat` and `Dog` show visible class-specific differences.
            - **Limitations:** a 10-epoch run may produce noisy or partial animals because GANs often need longer training and tuning for high-quality images.
            - **Possible improvements:** train longer, tune learning rate/batch size, increase model capacity, add augmentation, or use more advanced GAN losses.
